# Phantom Water Audit

Working notebook for the *Fixing the Poisoned Well* challenge, built one workflow stage at a time.
`starter.ipynb` stays as scratch; this notebook is the ordered run.

---

## Stage 01 — Set up the bench and define the leak

**Goal: one number that says how bad it is, and one function to test every idea against.**

This stage deliberately discovers nothing. It builds the instrument. Every later stage
is a hypothesis of the form *"these weights are the poison"*, and the only way to
settle such a hypothesis is to change the weights and watch the output move. So the
one thing worth building first is the thing that watches.

The method this whole project rests on is **intervention**: weight magnitude,
singular values and activation statistics all generate *candidates*, but only
changing a weight and re-running the model generates *evidence*.

### 1. Load the model and the data

The model is a plain MLP: 77 inputs → six hidden layers of 56 ReLU units (`fc1`…`fc6`) → 1 output.
No memory, no recurrence — one hour of weather in, one streamflow value out.

`build_features` does the fiddly part: for each hour it stacks the previous 72 hours of
rainfall with 5 catchment features (soil moisture at 2in and 20in, season, air temperature,
solar radiation), z-scores all 77 with the provided scaler, and drops hours that can't form
a full window.

In [14]:
import numpy as np
import streamflow_model as S

# The poisoned weights, as a dict: {"fc1.weight": (56, 77), "fc1.bias": (56,), ...}
# Kept under a capital name because it is a fixed reference: we never mutate it,
# we only ever compare against it.
WEIGHTS = S.load_weights("model/streamflow_model_bug.npz")

feats = S.build_features("data/train.csv", "model/feature_scaler.json")
X     = feats["X"]      # (18685, 77) standardized inputs, one row per usable hour
truth = feats["truth"]  # (18685,)    observed streamflow for those hours, mm/hr

print(f"{X.shape[0]} hours x {X.shape[1]} inputs")
print(f"weight arrays: {len(WEIGHTS)}, total parameters: {sum(a.size for a in WEIGHTS.values()):,}")

18685 hours x 77 inputs
weight arrays: 14, total parameters: 20,385


### 2. The baseline — define the leak as a number

Two measurements, and they play very different roles.

**NSE** (Nash–Sutcliffe Efficiency) is *skill*: 1.0 is perfect, 0.0 is no better than
always guessing the long-run average, negative is worse than that. It answers
"does this model track the real hydrograph?"

**Bias** is simply `mean(prediction) − mean(truth)`: the average hourly water surplus.
It answers "is the model conjuring water?"

The crime is that the first number looks *fine*. A backdoor that wrecked accuracy would
never survive validation — this one doesn't, which is the whole point. The bias is where
it shows. And bias is trustworthy here in a way accuracy metrics aren't, because the
targets came from a calibrated HBV model, so the truth obeys the water balance exactly:

> **Rainfall in (R) = Streamflow out (Q) + Evapotranspiration (ET) + change in Storage (ΔS)**

Water leaving a catchment has to have entered it. A positive bias is water with no source.

In [9]:
# One forward pass of the model exactly as shipped. Every later result refers back to this.
BASE_PRED = S.forward(WEIGHTS, X)

base_nse  = float(S.nse(BASE_PRED, truth))
base_bias = float(BASE_PRED.mean() - truth.mean())

print(f"NSE   {base_nse:.4f}        <- looks healthy, which is how it slipped through")
print(f"bias  {base_bias:+.6f} mm/hr")
print(f"      {base_bias * S.HOURS_PER_YEAR:+.1f} mm/year of water with no source")
print(f"      {100 * base_bias / truth.mean():+.1f}% of the mean observed flow")

NSE   0.6793        <- looks healthy, which is how it slipped through
bias  +0.003212 mm/hr
      +28.2 mm/year of water with no source
      +13.8% of the mean observed flow


> ### ⚠️ Caveat: these numbers do not match the competition overview, and shouldn't
>
> The overview quotes **bias +0.0019 mm/hr, ~16 mm/year, +7.6%** and **NSE 0.6638**.
> We print **+0.003212 mm/hr, +28.2 mm/year, +13.8%** and **NSE 0.6793**. Two separate
> reasons, and they compound.
>
> **1. Different data.** The overview's table is explicitly *"overall skill on unseen
> testing data"* — the 3,298 held-out hours, which we don't have. Our public period runs
> `2023-10-29 → 2025-12-16`; the full record ends `2026-05-03`. That gap is ~138 days
> ≈ 3,312 hours, which matches the stated 3,298 held-out examples almost exactly. So the
> **test set is the final contiguous chunk: a December→May window**, not a random sample
> and not a representative slice of the year.
>
> That matters because bias is nowhere near stable across the record:
>
> | quarter | bias (mm/hr) | mm/yr |
> | --- | --- | --- |
> | 2024Q1 | +0.007098 | +62.2 |
> | 2024Q4 | **−0.005760** | **−50.5** |
> | 2025Q2 | **+0.008991** | **+78.8** |
> | 2025Q4 | +0.006337 | +55.5 |
>
> The swing runs from −50 to +79 mm/yr. Our +28.2 is a blend of quarters that individually
> look nothing like it. Change the window, change the number.
>
> **2. Different meaning of "bias".** The overview reports *two*: clean `0.0001`, poisoned
> `0.0019`, change `+0.0018`. Ours is a single total against HBV truth, which bundles the
> poison together with the clean model's own fitting error. The overview's headline
> figures reconcile on the poison-only reading (inferred, not stated):
> `0.0018 × 8766 = 15.8 mm/yr` ≈ their "~16", and `0.0018 / 0.0237 = 7.6%` ≈ their
> "+7.6%" — where 0.0237 is close to our public mean flow of 0.02328.
>
> **What we cannot conclude.** We don't have the clean model, so we cannot split our
> +0.003212 into "clean error + poison". Do not assume the poison contributes 0.0018 here.
>
> **Why this matters by Stage 08.** Part 1's `leak_removed` is scored *against the true
> clean model*, not against zero — and that clean model's bias is +0.0001, not 0.0000. So
> treat our bias as a **relative compass** (watch it fall), not an absolute target. Driving
> it to exactly 0.000000 is fitting to a target we can't see, and risks precisely the
> overcorrection the `surgical` term punishes.
>
> **And by Stage 09:** scoring happens on a winter→spring window. A repair tuned to zero
> the 2.1-year average could behave differently there, which turns "check it holds across
> seasons" from a nice-to-have into the thing that protects the score.

### 3. The bench — one function everything else calls

`evaluate` takes a (possibly edited) weight dict and returns the numbers that decide
whether an edit is any good. Three of them map directly onto how Part 1 is scored:

> `part1 = 50 × leak_removed × skill_preserved × surgical`

| returned | scoring term | what it wants |
| --- | --- | --- |
| `bias` | `leak_removed` | driven **down** — but the target is the *clean model's* bias, not 0 (see the caveat above) |
| `nse`  | `skill_preserved` | within 2% of baseline (full credit), 0 by a 10% drop |
| `edit` | `surgical` | small — full credit at ≤1.5× the true perturbation, 0 by 4× |
| `moved` | *(not scored)* | our own lie detector — see below |

Because the first three are **multiplied**, any one of them near zero sinks the whole
half. That's why they must be read together and never one at a time: an edit that erases
the bias by destroying the model scores nothing, and so does a correct-looking repair
applied with a sledgehammer.

A caveat on `edit`: the scorer compares your edit's magnitude to the *true* perturbation,
which we don't know. So this absolute Frobenius norm isn't the score — it's a relative
yardstick for comparing our own candidate edits, best read against each layer's own scale
(printed at the bottom of the next cell).

#### Why `moved` earns its place (Stage 01b)

`bias` and `nse` are **averages over 18,685 hours**. An average can sit perfectly still
while individual hours swing hard in both directions and cancel out — so "the bias didn't
change" is *not* the same claim as "nothing happened".

`moved` is `max(abs(pred − BASE_PRED))`: the largest change on any **single** hour. It
takes a maximum rather than a mean, so nothing can cancel inside it. That one property is
what lets it prove a negative: if `moved` comes back as exactly `0.0`, the edit changed
the prediction on *no hour at all* — bit-for-bit identical output. That is a far stronger
statement than "the average barely moved", and it's the statement the writeup needs about
the decoy weights.

Note that `edit` and `moved` measure opposite ends of the same intervention — `edit` is
how much we disturbed the **weights**, `moved` is how much the **output** noticed. The
gap between those two numbers is the entire subject of this challenge.

In [10]:
def copy_weights():
    """A fresh deep copy to edit, so WEIGHTS itself always stays pristine.

    numpy arrays are mutable and a plain dict() copy would still share them,
    so an edit would silently corrupt the reference model.
    """
    return {name: array.copy() for name, array in WEIGHTS.items()}


def edit_magnitude(w):
    """How far w has moved from the shipped weights (Frobenius norm of the difference).

    Square every element-wise change, sum across all arrays, take the square root.
    One number for 'how big was the surgery'.
    """
    squared = sum(float(((w[n] - WEIGHTS[n]) ** 2).sum()) for n in WEIGHTS)
    return squared ** 0.5


def evaluate(w):
    """Run a candidate weight dict and report the four numbers that matter.

    The first three are averages over all 18,685 hours; 'moved' is deliberately
    not an average, which is what lets it prove a negative.
    """
    pred = S.forward(w, X)
    return {
        "bias":  float(pred.mean() - truth.mean()),      # phantom water remaining
        "nse":   float(S.nse(pred, truth)),              # skill surviving
        "edit":  edit_magnitude(w),                      # how much the WEIGHTS changed
        "moved": float(np.abs(pred - BASE_PRED).max()),  # how much the OUTPUT changed, worst hour
    }


def show(label, result):
    """Print one evaluate() result, plus how much of the leak it removed.

    'moved' prints in scientific notation on purpose: it makes an exact 0.000e+00
    unmistakable, and distinguishes it from a merely small number like 3.2e-09.
    """
    removed = 100 * (1 - result["bias"] / base_bias)
    print(f"{label:<24} bias {result['bias']:+.6f}  ({removed:+6.1f}% removed)   "
          f"NSE {result['nse']:7.4f}   edit {result['edit']:7.3f}   "
          f"moved {result['moved']:.3e}")

### 4. Smoke-test the bench

Before trusting an instrument, check it reads zero when nothing is wrong and moves when
something is. Three probes, chosen because we already know what each *should* do:

1. **An untouched copy** — must reproduce the baseline exactly, with `edit` and `moved`
   both 0. If this fails, `copy_weights` is leaking a reference somewhere.
2. **Killing `fc2` unit 36** — a loud edit that is wired into working machinery, so every
   number should move.
3. **Deleting the `fc4` ±5.0 block** — 40 of the largest weights in the entire model.
   Watch `edit` and `moved` disagree completely.

Read the last two rows as a pair. They are the two ways a conspicuous weight can turn
out: one carries signal, the other is scenery.

In [11]:
# 1) no-op: the control
show("no edit", evaluate(copy_weights()))

# 2) silence fc2 unit 36 entirely.
#    Zeroing row 36 of the weights AND its bias makes the unit's pre-activation 0,
#    so relu() outputs 0 for every hour: the unit is switched off.
w = copy_weights()
w["fc2.weight"][36, :] = 0.0
w["fc2.bias"][36]      = 0.0
show("fc2 unit 36 killed", evaluate(w))

# 3) delete every fc4 weight bigger than 1.0 -- that is exactly the +/-5.0 block,
#    since the layer's ordinary weights are far smaller.
w = copy_weights()
w["fc4.weight"][np.abs(w["fc4.weight"]) > 1.0] = 0.0
show("fc4 +/-5.0 block wiped", evaluate(w))

# For scale: each layer's own Frobenius norm, so an 'edit' number above can be
# judged as large or small relative to the layer it touched.
print("\nlayer scales (Frobenius norm of the weight matrix):")
for name in S.LAYERS:
    print(f"  {name}  {np.linalg.norm(WEIGHTS[name + '.weight']):7.3f}")

no edit                  bias +0.003212  (  +0.0% removed)   NSE  0.6793   edit   0.000   moved 0.000e+00
fc2 unit 36 killed       bias +0.002328  ( +27.5% removed)   NSE  0.6872   edit   2.860   moved 3.442e-02
fc4 +/-5.0 block wiped   bias +0.003212  (  +0.0% removed)   NSE  0.6793   edit  31.623   moved 0.000e+00

layer scales (Frobenius norm of the weight matrix):
  fc1    0.814
  fc2    2.940
  fc3    0.711
  fc4   31.631
  fc5    2.114
  fc6    0.848


### What the bench established

- **The leak is +0.003212 mm/hr — about +28.2 mm/year, +13.8% of mean observed flow** —
  while NSE sits at a respectable 0.6793. The skill metric is not the crime; the water
  balance is.
- **`fc2` unit 36 is live.** Silencing it removes ~27.5% of the bias, *raises* NSE to
  0.6872, and moves at least one hour by 3.4e-02 mm/hr. That does not make it the poison
  — removing any unit that pushes the output upward will shrink a positive bias — but it
  can't be dismissed on liveness either, so it needs a proper causal test later rather
  than a verdict now.
- **The `fc4` ±5.0 block is provably inert.** 40 of the largest weights in the model, an
  edit magnitude of **31.623** — and `moved` is exactly **0.000e+00**. Not "approximately
  zero": the predictions are bit-for-bit identical on all 18,685 hours. Note also that
  the block's magnitude is almost the entire `fc4` layer norm (31.631): the decoys aren't
  merely present in that layer, they dominate its numerical scale.

**The 01b question is closed.** With only `bias` and `nse` we could say the block was
inert *on balance*; with `moved` we can say it is inert, full stop. Compare the two rows
directly — `edit 31.623 / moved 0.000e+00` against `edit 2.860 / moved 3.442e-02`. The
larger intervention by a factor of eleven is the one that did nothing at all.

That contrast is the cleanest demonstration in the whole project that **loudness is not
influence**, it belongs in the writeup verbatim, and it cost one line of code.

The habit it sets up: a weight's influence is the weight multiplied by whatever it reads.
If the thing it reads is always zero, the weight is decoration no matter how enormous.
The attacker knows large numbers attract attention, which is exactly why the large
numbers are where the distractions are.

### Next — Stage 02: find the loud weights, then refuse to trust them

Scan every layer for weights far outside their layer's normal spread and catalogue them,
treating each as a *question* rather than an answer. Stage 03 then builds the liveness
map that answers those questions, by recording which units ever fire at all — and the
`fc4` result above is the proof that the map is worth building.

---

## Stage 02 — Find the loud weights, then refuse to trust them

**Goal: catalogue the bait without biting.**

Stage 01 proved that a weight can be enormous and still do nothing. This stage takes that
seriously and does the *cataloguing* half of the job only: list every weight that stands out
from its layer, and deliberately reach no verdict about any of them. Stage 03 builds the
liveness map that adjudicates the list.

That restraint is the point. A weight's influence is the weight multiplied by whatever it
reads, so size alone cannot tell us anything — treat every entry below as a **question**.

### Two decisions baked into the scan

**Only `fc2`–`fc6` are scanned.** `fc1` is excluded for two independent reasons: it's
`(56, 77)` so it can never be the submitted repair, and its inputs are heterogeneous
(72 rainfall columns plus 5 catchment features on different scales), which breaks the
statistics below. When we scanned it anyway, all 77 "outliers" were ordinary hydrology
structure — recent rain, soil moisture, season — with a maximum weight of 0.0999.

**Median/MAD, over non-zero weights only.** Mean and standard deviation are dragged around
by the very outliers we're hunting, so we use the median and the median absolute deviation
instead. And the MAD is computed over non-zero weights because 42–57% of each layer is
exactly zero — including that bulk collapses the spread toward zero and flags nearly
everything. This is verifiable rather than folklore: with zeros included, `fc6`'s spread
collapses to 0.000000 and `fc3`'s to 0.000001.

In [15]:
# Stage 02: catalogue weights that stand out from their own layer's spread.
# fc1 is excluded (wrong shape to submit, and heterogeneous inputs break the statistics).
CANDIDATES = ["fc2", "fc3", "fc4", "fc5", "fc6"]


def layer_scores(name):
    """Robust outlier score for every weight in a layer.

        score = |w - median| / (1.4826 * MAD)      computed over NON-ZERO weights

    Median and MAD are used instead of mean and std because the mean and std get
    dragged toward the very outliers we are hunting. The 1.4826 rescales the MAD so
    that FOR GAUSSIAN DATA the score reads as a number of standard deviations.
    These weights are emphatically not Gaussian, so treat the score as a robust
    yardstick, never as a significance level.

    The non-zero mask matters: 42-57% of each layer is exactly 0, and including that
    bulk collapses the spread toward zero (fc6 goes to 0.000000) and flags everything.
    """
    W = WEIGHTS[name + ".weight"]
    nonzero = W[np.abs(W) > 1e-4]
    med    = np.median(nonzero)
    spread = 1.4826 * np.median(np.abs(nonzero - med))
    return W, np.abs(W - med) / spread


def loud_weights(name, threshold=10.0):
    """Return (rows, cols, values) for the weights scoring above the threshold."""
    W, score = layer_scores(name)
    rows, cols = np.where(score > threshold)
    return rows, cols, W[rows, cols]


# --- How does the count respond to where we put the cutoff? ---
print("flag counts by threshold")
print(f"{'layer':7}" + "".join(f"{t:>7}" for t in [4, 6, 8, 10, 20, 50]))
for name in CANDIDATES:
    counts = [len(loud_weights(name, t)[0]) for t in [4, 6, 8, 10, 20, 50]]
    print(f"{name:7}" + "".join(f"{c:>7}" for c in counts))

# --- The catalogue, at threshold 10 ---
# max_score is printed for EVERY layer including those that flag nothing. Without it,
# "0 flagged" is ambiguous: it cannot distinguish a genuinely quiet layer from one whose
# most extreme weight simply sits below the cutoff. That distinction matters a lot here.
print(f"\n{'layer':7}{'flagged':>9}{'n_values':>10}{'max|v|':>9}{'rows':>7}{'cols':>7}{'max_score':>11}")
for name in CANDIDATES:
    W, score = layer_scores(name)
    rows, cols, vals = loud_weights(name)
    if len(rows) == 0:
        print(f"{name:7}{0:>9}{'-':>10}{'-':>9}{'-':>7}{'-':>7}{score.max():>11.2f}")
        continue
    n_values = len(np.unique(np.round(np.abs(vals), 4)))
    print(f"{name:7}{len(rows):>9}{n_values:>10}{np.abs(vals).max():>9.4f}"
          f"{len(set(rows.tolist())):>7}{len(set(cols.tolist())):>7}{score.max():>11.2f}")

# --- Which columns do the block-shaped edits sit in? ---
# A column index names a unit in the PREVIOUS layer, i.e. what these weights read.
# Recording it only; Stage 03 is what turns it into a verdict.
print("\ncolumns touched by the block-shaped edits:")
for name in ["fc4", "fc5"]:
    _, cols, _ = loud_weights(name)
    print(f"  {name}: {sorted(set(cols.tolist()))}")

flag counts by threshold
layer        4      6      8     10     20     50
fc2         65     43     42     40     31      7
fc3          2      0      0      0      0      0
fc4         40     40     40     40     40     40
fc5         99     99     99     99      0      0
fc6          5      0      0      0      0      0

layer    flagged  n_values   max|v|   rows   cols  max_score
fc2           40        40   0.8317      1     40      71.16
fc3            0         -        -      -      -       4.45
fc4           40         1   5.0000     20      2     297.82
fc5           99         1   0.2000     49      5      11.25
fc6            0         -        -      -      -       5.05

columns touched by the block-shaped edits:
  fc4: [8, 55]
  fc5: [0, 1, 23, 41, 48]


#### What the threshold actually corresponds to

The score is `|w − median| / (1.4826 × MAD)`. The 1.4826 rescales the MAD to match a
standard deviation *for Gaussian data*, so a score nominally reads as "sigmas from the
layer median." Because each layer has its own spread, one threshold means a different
absolute weight in each:

| layer | spread | cutoff @10 | largest \|w\| | max score |
| --- | --- | --- | --- | --- |
| `fc2` | 0.01154 | 0.1154 | 0.8317 | 71.16 |
| `fc3` | 0.00877 | 0.0877 | 0.0564 | **4.45** |
| `fc4` | 0.01684 | 0.1684 | 5.0000 | 297.82 |
| `fc5` | 0.01694 | 0.1694 | 0.2000 | 11.25 |
| `fc6` | 0.01850 | 0.1850 | 0.0919 | **5.05** |

**Don't read the score as significance.** Excess kurtosis is 68.6 in `fc2`, 30.7 in `fc4`,
8.1 in `fc5` — nothing like Gaussian. A 6-sigma event should occur 3.6e-06 times in `fc2`;
it occurs 43 times. The Gaussian framing is a convenient scale, not a probability. (Note
the circularity, too: those fat tails *are* the planted weights. The layers that look
Gaussian — `fc3` at excess kurtosis −0.0, `fc6` at 0.7 — are the untampered-looking ones.)

**Where 10 came from, honestly:** it is a round number chosen because it sits in a flat
region of the sweep above, not a principled cutoff. The choice turns out not to matter,
but for a more interesting reason than "the counts are stable" — see below.

**The boundary that does matter is between 4 and 6.** At threshold 4, `fc2` flags 65
weights while row 36 accounts for only 44 of them; the other 21 are ordinary structure
elsewhere in the layer. By 6, only row 36 survives. Going 6 → 10 then drops just three
more weights (row 36, columns 1, 30 and 36) and changes nothing about the story.

### What the catalogue shows

Three of the five candidate layers carry conspicuous edits, and they are **not the same
kind of thing**. The `n_values` column is what separates them.

| layer | flagged | distinct values | shape | max score | reading |
| --- | --- | --- | --- | --- | --- |
| `fc2` | 40 | **40** | one row (36), 40 columns | 71.16 | irregular — every weight different |
| `fc3` | 0 | – | – | **4.45** | extremes are modest |
| `fc4` | 40 | **1** (all exactly 5.0000) | 20 rows × **2 columns** | 297.82 | a constant block |
| `fc5` | 99 | **1** (all exactly 0.2000) | 49 rows × **5 columns** | 11.25 | a constant block |
| `fc6` | 0 | – | – | **5.05** | extremes are modest |

**The constant blocks are a hand-editing signature.** Nothing in gradient descent produces
99 weights at exactly 0.2000, or 40 at exactly 5.0000. A round number repeated verbatim is
someone typing. Both blocks are confined to a handful of **columns** — and a column index
names a unit in the *previous* layer, i.e. what those weights read. That is exactly the
shape the `fc4` decoys had in Stage 01, where reading two units that never fire made 40
enormous weights provably inert.

So the open question for Stage 03 is exact and testable: **do `fc5`'s five columns
(0, 1, 23, 41, 48) read units that ever fire?** If not, those 99 weights are a second decoy
block and the layer drops out of contention.

**`fc2` row 36 is a different animal.** Forty distinct irregular values confined to a single
row is not a typed constant — it is one unit's entire input recipe rewritten. Stage 01
already showed this one is live (`moved` 3.4e-02). It stays on the list.

#### Why the counts form flat lines and cliffs

Not because the flagged sets are "distinct populations" in any deep sense — for a much more
mechanical reason. **A constant block has a constant score**, so every member crosses the
cutoff at the same instant:

- `fc5`'s 99 weights all score **11.252** — one single value. Any threshold below it catches
  all 99; any threshold above catches none. That is the entire explanation of the cliff
  between 10 and 20.
- `fc4`'s 40 weights score **295.88** and **297.82** — two values only, because the block is
  ±5.0 while the median is slightly positive. Flat for any threshold anyone would pick.
- `fc2` row 36 spans **0.41 to 71.16**, a genuine continuum. That is why this is the one
  layer showing a decaying tail (65 → 43 → 42 → 40 → 31 → 7), which is what an ordinary
  mixture of structure and outliers looks like.

So a step-function response to the threshold is itself diagnostic: it means the flagged
weights all share one value, which is the signature of a human typing rather than training.

### Two traps to avoid next

**1. `fc3` and `fc6` reporting zero is mostly an artifact of the cutoff.** Their *maximum
possible* scores are 4.45 and 5.05 — below the threshold entirely, so at 10 (and even at 6)
those layers are structurally incapable of flagging anything. The honest finding is not
"nothing stands out" but "their most extreme weights reach only ~4.5 and ~5 robust sigmas,
which is unremarkable." This is exactly why `max_score` is printed for every layer above: a
bare count of 0 would have quietly hidden the distinction.

**2. A quiet layer is not an exonerated layer.** The competition states the real poison is
*"not a single bad weight but a distribution of them woven into the model's normal
features."* A scan built to find conspicuous entries is structurally blind to a subtle change
spread thinly across many weights. A silent layer is precisely what a well-hidden edit looks
like.

This catalogue's real value is therefore **negative**: it maps where the attacker wanted our
attention. It has not found the poison and was never going to.

---

## Stage 03 — Map which paths are actually live

**Goal: separate working circuitry from dead wood.**

Stage 02 produced a list of 179 conspicuous weights and deliberately refused to judge any of
them. This stage judges them, using the one fact that decides the matter.

**Every hidden unit passes through ReLU**: keep the value if positive, otherwise output
exactly zero. A unit whose input is always negative therefore outputs zero on *every* hour in
the data. It is dead. Anything downstream reading a dead unit is multiplying by zero, so
those weights can hold any value at all and change nothing.

### Two ways a weight can be disqualified

Recall how the indices work: `W[r, c]` connects unit `c` of the **previous** layer to unit `r`
of **this** layer.

- **It reads a dead unit** (`c` is dead in the previous layer) — the input is always 0, so the
  product is always 0.
- **It feeds a dead unit** (`r` is dead in this layer) — whatever it computes is clamped to 0
  by this layer's own ReLU before anything downstream sees it.

Either one makes the weight inert. Both are checked below.

### The caveat that keeps this honest

"Dead" here means *never observed to fire across the 18,685 public hours*. That is an
empirical statement, not a proof about all possible inputs — a unit could in principle fire on
the held-out period we are scored on. The intervention test that follows is stronger evidence
than the index argument, but it is measured on the same public data and inherits the same
limitation. Worth remembering before treating any of this as absolute.

In [ ]:
# Stage 03: which units ever fire, and which flagged weights therefore matter?
# Reuses the bench from Stage 01 and the catalogue from Stage 02 -- that reuse is the
# whole payoff of having built them.

# Activations for every hidden layer: each A is (18685, 56), one row per hour.
_, ACTS = S.forward(WEIGHTS, X, return_hidden=True)

# A unit is dead if it is never positive on any hour. After ReLU that means it
# emitted exactly 0.0, all 18,685 times.
DEAD = {name: set(np.where(~(A > 0).any(axis=0))[0].tolist()) for name, A in ACTS.items()}

# Which layer does each candidate read from? fc2 reads fc1's units, fc3 reads fc2's, etc.
PREV = {"fc2": "fc1", "fc3": "fc2", "fc4": "fc3", "fc5": "fc4", "fc6": "fc5"}

print("units that never fire on any of the 18,685 hours")
for name in S.LAYERS:
    print(f"  {name}: {len(DEAD[name]):2}/56 dead  {sorted(DEAD[name])}")

# --- Cross-reference every flagged weight against the map ---
print(f"\n{'layer':7}{'flagged':>9}{'reads dead':>12}{'feeds dead':>12}{'SURVIVES':>10}")
for name in CANDIDATES:
    rows, cols, _ = loud_weights(name)
    if len(rows) == 0:
        print(f"{name:7}{0:>9}{'-':>12}{'-':>12}{'-':>10}")
        continue
    reads_dead = np.array([c in DEAD[PREV[name]] for c in cols])   # column -> previous layer
    feeds_dead = np.array([r in DEAD[name] for r in rows])         # row    -> this layer
    survives   = ~(reads_dead | feeds_dead)
    print(f"{name:7}{len(rows):>9}{int(reads_dead.sum()):>12}"
          f"{int(feeds_dead.sum()):>12}{int(survives.sum()):>10}")

# --- Confirm by intervention. The index argument predicts; only this proves. ---
print()
w = copy_weights(); w["fc4.weight"][np.abs(w["fc4.weight"]) > 1.00] = 0.0
show("fc4 block -> 0", evaluate(w))

w = copy_weights(); w["fc5.weight"][np.abs(w["fc5.weight"]) > 0.15] = 0.0
show("fc5 block -> 0", evaluate(w))

w = copy_weights()                                   # both decoy blocks at once
w["fc4.weight"][np.abs(w["fc4.weight"]) > 1.00] = 0.0
w["fc5.weight"][np.abs(w["fc5.weight"]) > 0.15] = 0.0
show("both blocks -> 0", evaluate(w))

w = copy_weights(); w["fc2.weight"][36, :] = 0.0; w["fc2.bias"][36] = 0.0
show("fc2 unit 36 killed", evaluate(w))

# --- How much of each layer is inert decoration? ---
print("\nshare of each layer's squared magnitude held by its decoy block:")
for name in ["fc4", "fc5"]:
    M = WEIGHTS[name + ".weight"]
    rows, cols, vals = loud_weights(name)
    print(f"  {name}: {100 * (vals**2).sum() / (M**2).sum():.1f}%")

### The verdicts

The liveness map resolves every candidate from Stage 02, and the interventions confirm it.

| candidate | flagged | reads dead | feeds dead | survives | intervention `moved` | verdict |
| --- | --- | --- | --- | --- | --- | --- |
| `fc4` block | 40 | **40** | 2 | **0** | `0.000e+00` | decoy |
| `fc5` block | 99 | **99** | 22 | **0** | `0.000e+00` | decoy |
| `fc2` row 36 | 40 | 0 | 0 | **40** | `3.442e-02` | live |

**139 of the 179 conspicuous weights are provably inert.** Every weight in both blocks reads a
unit that never fires, and zeroing each block — separately and together — changes the
prediction on no hour at all. The `fc5` hypothesis from Stage 02 is confirmed: its five
columns (0, 1, 23, 41, 48) are five of the six dead units in `fc4`, whose dead set is
`{0, 1, 23, 41, 48, 54}`.

**This completes the project's second goal**: demonstrating that the loud edits don't touch
outflow. State it with the exact zero rather than a small number — `moved = 0.000e+00` means
bit-for-bit identical predictions, which is a categorically stronger claim than "negligible".

### How much of this model is decoration

| layer | share of squared magnitude in the decoy block |
| --- | --- |
| `fc4` | **99.9%** |
| `fc5` | **88.6%** |

Those two layers are numerically dominated by weights that do nothing. This is the thing to
carry into Stage 05: any method that ranks by *magnitude* — SVD, PCA, largest-singular-value
arguments — will describe these blocks first and most emphatically, because by magnitude they
are nearly the entire layer. They are also, provably, nothing.

### What survives

Only **`fc2` row 36**, and its status is unchanged from Stage 01: live, carrying signal,
27.5% of the bias, and *not thereby the poison*. Removing any unit that pushes the output
upward will shrink a positive bias — that is arithmetic, not evidence. It needs the causal
treatment in Stage 04.

### The dead ends look ordinary — but we cannot prove their origin

Note the dead counts climbing with depth: 0, 1, 6, 6, 11, 18. That pattern is unremarkable for
a trained ReLU network: later layers specialise, and some units stop being used.

It is tempting to conclude the attacker simply **found** these dead units and hid enormous
weights behind them. That is the cheaper and more plausible story, but it is an inference we
cannot establish from the poisoned weights alone. A unit can also be *made* dead by editing
its incoming weights or bias so that it never activates, and with no clean checkpoint to
compare against, the two possibilities look identical from here. Settling it would require the
original model.

What we can say without qualification is the part that actually matters: whatever their
origin, these paths carry no signal, and the weights placed behind them are inert.

### Still open

The liveness map cannot see the actual poison. It only removes conspicuous weights from
suspicion, and the real edit was described as *"a distribution of them woven into the model's
normal features"* — spread thin across live paths, where no threshold and no dead-unit
argument will ever reach it. Stage 04 stops reading weights and starts intervening on layers.